In [1]:
! pip install -U sentence-transformers
! pip install -U chromadb


In [2]:
# 1️⃣ Importar librerías
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import os
import numpy as np
import uuid



In [3]:
# Modelo de embeddings
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

def get_embeddings(text):
    return model.encode(text).tolist()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [54]:
# Inicializar ChromaDB con persistencia
client = chromadb.PersistentClient(path="./chroma_store")
collection = client.get_or_create_collection(name="documentos_ia")

In [55]:
def cargar_documentos():
    docs_path = "/content/drive/MyDrive/Reto_Qubika_Icesi/Qubika/notebooks/docs"
    documentos = []
    metadatos = []
    ids = []

    for filename in os.listdir(docs_path):
        if filename.endswith(".txt"):
            with open(os.path.join(docs_path, filename), 'r', encoding="utf-8") as f:
                text = f.read().strip()
                chunks = [text[i:i+500] for i in range(0, len(text), 500)]

                for i, chunk in enumerate(chunks):
                    documentos.append(chunk)
                    metadatos.append({"source": filename})
                    ids.append(str(uuid.uuid4()))  # ID único

    return documentos, metadatos, ids

In [7]:

def crear_base_datos():
    docs, metas, ids = cargar_documentos()
    embeddings = [get_embeddings(d) for d in docs]
    collection.add(documents=docs, embeddings=embeddings, metadatas=metas, ids=ids)
    print("✅ Base de datos creada con éxito.")

# --- CRUD FUNCIONES ---

def create_example(new_doc):
    """Añadir un documento nuevo"""
    embedding = get_embeddings(new_doc)
    doc_id = str(uuid.uuid4())
    metadata = {"source": "nuevo_documento.txt"}

    collection.add(
        documents=[new_doc],
        embeddings=[embedding],
        metadatas=[metadata],
        ids=[doc_id]
    )
    print(f"✅ Documento añadido con ID: {doc_id}")

def read_example(query):
    """Consultar documentos similares"""
    query_emb = get_embeddings(query)
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=1
    )

    if results["documents"]:
        print(f"📄 Mejor coincidencia: {results['documents'][0][0]}")
        print(f"📁 Fuente: {results['metadatas'][0][0]['source']}")
        print(f"🆔 ID: {results['ids'][0][0]}")
    else:
        print("⚠️ No se encontraron resultados.")

def get_documents_by_id(doc_id):
    """Obtener un documento por ID"""
    results = collection.get(ids=[doc_id], include=["documents", "metadatas", "embeddings"])

    if results["documents"]:
        print(f"📄 Documento: {results['documents'][0]}")
        print(f"📁 Fuente: {results['metadatas'][0]['source']}")
    else:
        print("⚠️ Documento no encontrado.")

def update_example(old_id, new_text):
    """Actualizar un documento (eliminar y reinsertar)"""
    collection.delete(ids=[old_id])

    new_id = str(uuid.uuid4())
    embedding = get_embeddings(new_text)
    metadata = {"source": "documento_actualizado.txt"}

    collection.add(
        documents=[new_text],
        embeddings=[embedding],
        metadatas=[metadata],
        ids=[new_id]
    )
    print(f"♻️ Documento actualizado. Nuevo ID: {new_id}")

def delete_example(doc_id):
    """Eliminar por ID"""
    collection.delete(ids=[doc_id])
    print(f"🗑️ Documento con ID '{doc_id}' eliminado.")

In [56]:
crear_base_datos()

✅ Base de datos creada con éxito.


In [9]:
get_embeddings("Reynaldo González es un ingeniero de software con 10 años de experiencia en desarrollo web y móvil.")


[-0.09879042953252792,
 0.011900155805051327,
 -0.07934798300266266,
 -0.07173145562410355,
 0.04849895089864731,
 -0.08828748762607574,
 0.04442288354039192,
 0.06878754496574402,
 -0.044716618955135345,
 -0.019074363633990288,
 0.00939092319458723,
 0.0556383952498436,
 0.020385727286338806,
 0.005857592914253473,
 0.05711750686168671,
 0.03409786522388458,
 -0.033253468573093414,
 0.008881228975951672,
 0.0223626010119915,
 -0.0675140768289566,
 0.11720021069049835,
 -0.029926331713795662,
 0.0018127127550542355,
 -0.00767973717302084,
 -0.014972873032093048,
 0.0011125488672405481,
 0.006905650720000267,
 -0.01639213040471077,
 -0.025348832830786705,
 -0.09709929674863815,
 0.034837014973163605,
 0.06626580655574799,
 0.07979561388492584,
 0.05058012530207634,
 -0.024885067716240883,
 -0.004187214653939009,
 0.021157726645469666,
 -0.058077357709407806,
 -0.07937496900558472,
 0.01205083541572094,
 -0.1423080712556839,
 0.029957251623272896,
 -0.0037484909407794476,
 -0.07423307746

In [10]:
results = collection.get(include=["embeddings", "documents", "metadatas"])

In [11]:
for id, content, metadata, embedding in zip(results["ids"], results["documents"], results["metadatas"], results["embeddings"]):
    print(f"ID: {id}, Contenido: {content[:100]} ", f"Metadata: {metadata}", f"Embedding: {embedding[:5]}...")


ID: c116f1fc-55ec-42f8-acb6-321381c91ec4, Contenido: El sol se alzaba sobre las colinas de Buenos Aires, pintando el cielo con tonos naranja y rosa. El a  Metadata: {'source': 'doc1.txt'} Embedding: [ 0.05201021 -0.04184366  0.03102788  0.03516816  0.0039017 ]...
ID: cb77d48f-abe9-4c1e-bc33-481b50560bb8, Contenido: ciosa continuaba, con el tráfico constante y el murmullo de las conversaciones. Los turistas se mara  Metadata: {'source': 'doc1.txt'} Embedding: [ 0.06561408  0.0027415   0.01852138 -0.00384711 -0.08065528]...
ID: bb495df8-1276-4fdb-b00f-2a5c58c94c2b, Contenido:  los restaurantes servían deliciosas parrilladas.

Hacia el norte, el elegante barrio de Recoleta in  Metadata: {'source': 'doc1.txt'} Embedding: [ 0.06329249 -0.05760499  0.0101936   0.01427128 -0.11836303]...
ID: 6064287c-c3d5-42d8-8cd1-f1bcbcd9efab, Contenido: y novelas exploran la realidad, la fantasía y la condición humana con una prosa exquisita.

El fútbo  Metadata: {'source': 'doc1.txt'} Embedding: [ 0.10750

In [14]:
while True:
    opcion = input("Elige una opción o escribe ayuda: ")

    if opcion=="ayuda":
        print("\n--- Menú CRUD ---")
        print("1. Crear documento")
        print("2. Consultar por tema")
        print("3. Actualizar documento (por ID)")
        print("4. Eliminar documento (por ID)")
        print("5. Buscar documento (por ID)")
        print("6. Salir")

    if opcion == "1":
        nuevo_doc = input("Ingresa el texto a añadir: ")
        create_example(nuevo_doc)

    elif opcion == "2":
        consulta = input("Consulta: ")
        read_example(consulta)

    elif opcion == "3":
        doc_id = input("ID del documento a actualizar: ")
        nuevo_texto = input("Nuevo contenido: ")
        update_example(doc_id, nuevo_texto)

    elif opcion == "4":
        doc_id = input("ID del documento a eliminar: ")
        delete_example(doc_id)

    elif opcion == "5":
        doc_id = input("ID del documento a buscar: ")
        get_documents_by_id(doc_id)

    elif opcion == "6":
        print("👋 Saliendo del programa.")
        break

Elige una opción o escribe ayuda: 1
Ingresa el texto a añadir: La gravedad es una fuerza fundamental en el universo
✅ Documento añadido con ID: b4a6d27e-238b-42d1-8abb-c0363abbd4f7
Elige una opción o escribe ayuda: 2
Consulta: "Fuerzas naturales que gobiernan el cosmos
📄 Mejor coincidencia: territorio y sus influencias culturales.

La ciencia y la tecnología también tienen un lugar importante en Argentina. Investigadores y científicos trabajan en diversas áreas, desde la biotecnología hasta la astronomía, contribuyendo al avance del conocimiento.

La música folklórica argentina, con sus ritmos y melodías características, cuenta historias de la tierra y sus habitantes. Instrumentos como la guitarra, el charango y el bombo legüero son protagonistas de estas expresiones culturales.

E
📁 Fuente: doc1.txt
🆔 ID: 9ae7609b-d645-42f1-930b-f9c784526c83
Elige una opción o escribe ayuda: 6
👋 Saliendo del programa.


In [15]:
collection.peek()

{'ids': ['c116f1fc-55ec-42f8-acb6-321381c91ec4',
  'cb77d48f-abe9-4c1e-bc33-481b50560bb8',
  'bb495df8-1276-4fdb-b00f-2a5c58c94c2b',
  '6064287c-c3d5-42d8-8cd1-f1bcbcd9efab',
  '9ae7609b-d645-42f1-930b-f9c784526c83',
  'da2355e1-112d-4490-914c-0af4e9d3b3ff',
  'ffeb1762-e361-42a0-8b51-e4288108e784',
  '35fc8847-96ed-4b63-9121-51e96ef4f211',
  '14e803d6-1473-4a75-acca-5a79103e1828',
  '1484b95c-5af1-44c3-a8ac-079f427c476b'],
 'embeddings': array([[ 0.05201021, -0.04184366,  0.03102788, ...,  0.0215565 ,
          0.0045086 , -0.0563845 ],
        [ 0.06561408,  0.0027415 ,  0.01852138, ...,  0.13028231,
          0.02773693, -0.06095771],
        [ 0.06329249, -0.05760499,  0.0101936 , ...,  0.06882158,
          0.01307632, -0.08279739],
        ...,
        [ 0.02917658, -0.00553686, -0.05402182, ...,  0.11687937,
          0.07559912, -0.03764471],
        [ 0.02340962, -0.03779529, -0.00181252, ..., -0.00685355,
          0.07568599, -0.06990532],
        [ 0.06885115, -0.03071461, 


**Punto 1**
*   ¿Qué tipo de operación CRUD usaste primero?
Se uso Create. Para eso se chunkea el texto y se hace su representación en embeddings y se inserta en chroma
*   ¿Cuál es el ID del documento creado? (Usa collection.peek())
14e803d6-1473-4a75-acca-5a79103e1828'

Explica por qué el documento aparece como coincidencia en la consulta.


**Punto 2**

In [17]:
create_example("Redes neuronales imitan procesos cerebrales")

✅ Documento añadido con ID: 49063759-2389-49c2-aafb-28901f602e05


In [18]:
update_example("49063759-2389-49c2-aafb-28901f602e05", "Deep Learning es un subcampo de IA con redesneuronales profundas")

♻️ Documento actualizado. Nuevo ID: 16cbc850-d8be-4da5-8a08-3f697e545702


In [19]:
delete_example("16cbc850-d8be-4da5-8a08-3f697e545702")

🗑️ Documento con ID '16cbc850-d8be-4da5-8a08-3f697e545702' eliminado.


In [20]:
update_example("16cbc850-d8be-4da5-8a08-3f697e545702", "Deep Learning es un subcampo de IA con redesneuronales profundas")

♻️ Documento actualizado. Nuevo ID: 134d8deb-16bd-4071-95e2-25a54aecaa3a


*  ¿Qué ocurre si intentas actualizar un documento eliminado?
Se vuelve a crear y le asigna un nuevo ID,
*  ¿Cómo se garantiza la integridad de los datos en las operaciones?
Asignando Id únicos a cada embedding o documento, lo que garantiza transacciones limpias y trazabilidad de las mismas sin duplicados.

**Punto 3**

In [22]:
delete_example("16cbc850-d8be-4da5-8a08-3f697e5452")

🗑️ Documento con ID '16cbc850-d8be-4da5-8a08-3f697e5452' eliminado.


In [25]:
new_doc = ""
embedding = get_embeddings(new_doc)
doc_id = str(uuid.uuid4())
metadata = {"source": "nuevo_documento.txt"}
collection.add(
        documents=[new_doc],
        embeddings=[embedding],
        metadatas=[metadata],
        ids=[doc_id]
    )

**punto4**

In [27]:
create_example("Python es ideal para ciencia de datos")

✅ Documento añadido con ID: af77e71c-f8a2-4f08-949d-9e830f2c3aa4


In [28]:
create_example("JavaScript maneja interacciones web dinámicas")

✅ Documento añadido con ID: 6a4f6c23-3fc3-49bc-9394-3c41774fd427


In [29]:
create_example("Java se usa en aplicaciones empresariales")

✅ Documento añadido con ID: 017eea2d-94bb-4530-bd30-610b95a85bdd


In [30]:
read_example("Lenguajes backend y frontend")

📄 Mejor coincidencia: lenguaje natural (PNL) es un campo de la inteligencia artificial que se centra en la comprensión y generación del lenguaje humano. Los embeddings son una herramienta fundamental en muchas tareas de PNL.

Los sistemas de recomendación utilizan bases de datos vectoriales para encontrar elementos similares a los que un usuario ha mostrado interés previamente. Esto permite ofrecer sugerencias personalizadas de productos, películas o música.

La detección de anomalías se basa en la identificación de 
📁 Fuente: doc1.txt
🆔 ID: 14e803d6-1473-4a75-acca-5a79103e1828


**  ¿Cuáles documentos aparecen como coincidencias?

Aparece como coincidencia el documento 1.

**  Analiza cómo las palabras clave influyen en los resultados.
Las palabras frontend y backend no tenían referentes en la base de datos. Por lo tanto la búsqueda se decanto por la palabra lenguaje y dio como resultado el texto cuyo tema es lenguaje natural.

**punto 5**

In [31]:
create_example("Ecologia es cuidar la naturaleza")

✅ Documento añadido con ID: 8f2305b7-2f4e-451c-b50c-f24b95612a5d


In [32]:
create_example("La mineria de datos es un area de la inteligencia artificial")

✅ Documento añadido con ID: 23af8c7b-90f9-441f-8b68-8ae3ab0d7752


In [34]:
collection.delete(ids=["8f2305b7-2f4e-451c-b50c-f24b95612a5d"])

In [36]:
collection.get(ids=["8f2305b7-2f4e-451c-b50c-f24b95612a5d"])


{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

Si busco un texto que se haya borrado da vacia la consulta

**punto 6**

In [39]:
create_example("IA y ética son temas actuales")

✅ Documento añadido con ID: 96f17335-8bc8-40f1-9e72-b779841ff65e


In [40]:
update_example("49063759-2389-49c2-aafb-28901f602e05", "Deep Learning es un subcampo de IA con redesneuronales profundas")


♻️ Documento actualizado. Nuevo ID: c803a44d-141a-4f3f-8ef9-ba7fd1e90256


El update está diseñado para borrar el docuemtento anterior y crear uno nuevo con un nuevo  id. Como se ve en la función.


```
def update_example(old_id, new_text):
    """Actualizar un documento (eliminar y reinsertar)"""
    collection.delete(ids=[old_id])
    
    new_id = str(uuid.uuid4())
    embedding = get_embeddings(new_text)
    metadata = {"source": "documento_actualizado.txt"}
    
    collection.add(
        documents=[new_text],
        embeddings=[embedding],
        metadatas=[metadata],
        ids=[new_id]
```



**Punto 7**

In [41]:
create_example("Blockchain garantiza transacciones seguras")

✅ Documento añadido con ID: fac1ba58-11c4-4d4e-bc74-99149eabe749


In [42]:
create_example("Criptomonedas usan tecnología blockchain")

✅ Documento añadido con ID: f4a62616-512b-4a20-a133-20005dfc6593


In [43]:
read_example("Seguridad en finanzas digitales")

📄 Mejor coincidencia: Blockchain garantiza transacciones seguras
📁 Fuente: nuevo_documento.txt
🆔 ID: fac1ba58-11c4-4d4e-bc74-99149eabe749


La búsqueda se decanta por la simililaridad con la palabra seguriddad, es decir, que la representación numerica es más cercana.

**Punto8**

In [59]:
results = collection.get()

# Obtener los IDs de los documentos
ids = results["ids"]
ids

['646dbaf6-9f3c-451b-afbb-a398e5544f17',
 '87e32ffe-e64b-42a6-9f04-2e0094f5c64a',
 '13c2d75a-73ac-45b1-9fd5-bf9b756bced3',
 '9010b1fc-6c71-4e6b-b3a1-830cf7452a9b',
 '0c7b1b62-099e-4e46-ade5-be34399576ab',
 '7265bddd-44ab-4f00-8b86-0bf032def927',
 '232e0aeb-d43a-459c-9364-2a01e18fce08',
 '1b313ab8-4980-4292-a3d8-ba23586533c1',
 '683d50ba-7cd3-40df-8d5e-374965f34e1d',
 'fa49f4b4-332e-4079-abb7-628b81b7cb66',
 'f6cd89c8-8c2b-40b4-8e9e-36791d6608f9',
 'a9b0766b-c0ba-4764-abdd-d24ccb3512a6',
 '296bc93a-9bd6-4649-b988-592234ca0325',
 '79f10fd9-a88f-4c92-b9fc-98c4cb3eacb6',
 '60c67cec-a63c-4137-a930-22efd42ac438',
 'b93f20b2-3ca2-4787-a809-28bf7b8dcf18',
 '39000c0d-8fa3-4c35-8116-80dcdf016944',
 '1187f90d-0dbc-4495-a356-10891559d179',
 'e82d7936-0c75-45a7-9e35-0863cd0ccaa1',
 'fc137174-ba28-4273-8f0d-cb85ac843249',
 'a713e947-acb9-490d-903f-e38cc7878a6b',
 '8161ec40-4ca9-4c2b-9923-763a2bcd48fe',
 '1b95b208-f370-4ff4-a3c9-d51f3371b5ec',
 '9d68010b-6c69-4af7-ab0a-0b0b13249d06',
 '4f08ce21-74b8-

In [60]:
collection.delete(ids=ids)

In [61]:
collection.count()


0

In [62]:
crear_base_datos()

✅ Base de datos creada con éxito.


In [63]:
collection.count()

33

**Punto 9**

In [66]:
new_doc =''
embedding = get_embeddings(new_doc)
doc_id = str(uuid.uuid4())
metadata = {"source": "nuevo_documento.txt"}

collection.add(
        documents=[new_doc],
        embeddings=[embedding],
        metadatas=[metadata],
        ids=[doc_id]
    )